# 09_evaluation.ipynb

## Project: Fabric Defect Detection
### Part 9: Comprehensive Evaluation (Custom Implementation)

This notebook loads the pre-trained models (from Notebook 7 & 8) and performs a final evaluation on the Test Set.

**Note:** All evaluation metrics and plotting logic are implemented from scratch using `numpy` and `matplotlib`.

In [1]:
# 1. SETUP & GOOGLE DRIVE
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pickle
import json
import glob
import shutil
from datetime import datetime
from google.colab import drive

# Mount Drive
if os.path.ismount('/content/drive'):
    print("✓ Drive already mounted.")
else:
    drive.mount('/content/drive', force_remount=True)

# Define Paths
BASE_DIR = "/content/drive/MyDrive/FabricDefect"
FEATURE_DIR = f"{BASE_DIR}/features"
MODEL_DIR = f"{BASE_DIR}/models"
BACKUP_DIR = f"{BASE_DIR}/backups"
METRICS_DIR = f"{BASE_DIR}/metrics"

os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)

# Plotting Config
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

print("✓ Google Drive mounted")
print("✓ Notebook directories ready")

Mounted at /content/drive
✓ Google Drive mounted
✓ Notebook directories ready


## 2. CUSTOM METRICS IMPLEMENTATION (No Scikit-Learn)

Implementing math for Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and ROC/AUC using NumPy.

In [2]:
class CustomEvaluator:
    @staticmethod
    def accuracy(y_true, y_pred):
        return np.mean(y_true == y_pred)

    @staticmethod
    def confusion_matrix(y_true, y_pred, num_classes):
        """Generates KxK confusion matrix from scratch"""
        cm = np.zeros((num_classes, num_classes), dtype=int)
        for t, p in zip(y_true, y_pred):
            cm[t][p] += 1
        return cm

    @staticmethod
    def classification_report(y_true, y_pred, class_names):
        """Calculates Precision, Recall, F1 for each class and averages"""
        num_classes = len(class_names)
        cm = CustomEvaluator.confusion_matrix(y_true, y_pred, num_classes)

        report = {}
        precisions = []
        recalls = []
        f1s = []
        supports = []

        print(f"{'Class':<20} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Support':<10}")
        print("-" * 65)

        for i in range(num_classes):
            tp = cm[i, i]
            fp = np.sum(cm[:, i]) - tp
            fn = np.sum(cm[i, :]) - tp
            support = np.sum(cm[i, :])

            # Prevent div by zero
            p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0

            report[class_names[i]] = {
                'precision': p, 'recall': r, 'f1-score': f1, 'support': support
            }

            precisions.append(p)
            recalls.append(r)
            f1s.append(f1)
            supports.append(support)

            print(f"{class_names[i]:<20} {p:.4f}     {r:.4f}     {f1:.4f}     {support}")

        # Macro Avg
        macro_p = np.mean(precisions)
        macro_r = np.mean(recalls)
        macro_f1 = np.mean(f1s)

        # Weighted Avg
        total_support = np.sum(supports)
        weighted_p = np.sum(np.array(precisions) * np.array(supports)) / total_support
        weighted_r = np.sum(np.array(recalls) * np.array(supports)) / total_support
        weighted_f1 = np.sum(np.array(f1s) * np.array(supports)) / total_support

        print("-" * 65)
        print(f"{'Macro Avg':<20} {macro_p:.4f}     {macro_r:.4f}     {macro_f1:.4f}     {total_support}")
        print(f"{'Weighted Avg':<20} {weighted_p:.4f}     {weighted_r:.4f}     {weighted_f1:.4f}     {total_support}")

        return {
            'per_class': report,
            'macro_avg': {'precision': macro_p, 'recall': macro_r, 'f1': macro_f1},
            'weighted_avg': {'precision': weighted_p, 'recall': weighted_r, 'f1': weighted_f1},
            'accuracy': np.trace(cm) / np.sum(cm)
        }

    @staticmethod
    def roc_curve_manual(y_true, y_score):
        """Calculates TPR and FPR for binary classification"""
        desc_score_indices = np.argsort(y_score)[::-1]
        y_score_desc = y_score[desc_score_indices]
        y_true_desc = y_true[desc_score_indices]

        distinct_value_indices = np.where(np.diff(y_score_desc))[0]
        threshold_idxs = np.r_[distinct_value_indices, y_true_desc.size - 1]

        tps = np.cumsum(y_true_desc)[threshold_idxs]
        fps = (1 + threshold_idxs) - tps

        total_pos = np.sum(y_true)
        total_neg = len(y_true) - total_pos

        if total_pos == 0: return [0.0, 1.0], [0.0, 1.0], 0.0
        if total_neg == 0: return [0.0, 1.0], [0.0, 1.0], 0.0

        tpr = np.r_[0, tps / total_pos]
        fpr = np.r_[0, fps / total_neg]

        auc = np.trapz(tpr, fpr)
        return fpr, tpr, auc

print("✓ COMPREHENSIVE METRICS CALCULATOR DEFINED")

✓ COMPREHENSIVE METRICS CALCULATOR DEFINED


## 3. MODEL CLASS DEFINITIONS
These definitions are required for `pickle` to successfully load the objects saved in previous notebooks.

In [3]:
import numpy as np

# Helper
def softmax(z):
    z -= np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

# --- Base Tree Components ---
class TreeNode:
    def __init__(self, depth=0):
        self.depth = depth
        self.left = None
        self.right = None
        self.feature_idx = None
        self.threshold = None
        self.value = None
        self.is_leaf = False

class RegularizedTree:
    def __init__(self, max_depth=3, lambda_reg=1.0, gamma=0.0, min_child_weight=1.0, feature_subsample=1.0):
        self.max_depth = max_depth
        self.lambda_reg = lambda_reg
        self.gamma = gamma
        self.root = None
    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])
    def _traverse(self, x, node):
        if node.is_leaf: return node.value
        if x[node.feature_idx] <= node.threshold: return self._traverse(x, node.left)
        else: return self._traverse(x, node.right)

# --- Model Wrappers ---
class CustomModelBase:
    def predict_proba(self, X):
        if not hasattr(self, 'n_classes'): self.n_classes = len(self.trees[0])
        scores = np.zeros((X.shape[0], self.n_classes))
        for iter_trees in self.trees:
            for k, tree in enumerate(iter_trees):
                scores[:, k] += self.lr * tree.predict(X)
        return softmax(scores)
    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

class CustomXGBoost(CustomModelBase):
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3, reg_lambda=1.0, gamma=0.0):
        self.K = n_estimators
        self.lr = learning_rate
        self.trees = []

class CustomLightGBM(CustomModelBase):
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3, goss_alpha=0.2, goss_beta=0.2):
        self.K = n_estimators
        self.lr = learning_rate
        self.trees = []

class CustomCatBoost(CustomModelBase):
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3):
        self.K = n_estimators
        self.lr = learning_rate
        self.trees = []

# --- Ensemble Stacker ---
class WeightOptimizationStacker:
    def __init__(self, n_trials=5000):
        self.n_trials = n_trials
        self.best_weights = None

    def predict(self, preds_list):
        # preds_list is a list of probability arrays [ [N, K], [N, K], ... ]
        weighted_sum = np.zeros_like(preds_list[0])
        for p, w in zip(preds_list, self.best_weights):
            weighted_sum += p * w
        return np.argmax(weighted_sum, axis=1)

    def predict_proba(self, preds_list):
        weighted_sum = np.zeros_like(preds_list[0])
        for p, w in zip(preds_list, self.best_weights):
            weighted_sum += p * w
        return weighted_sum

print("✓ ENSEMBLE & MODEL CLASS DEFINITIONS LOADED")

✓ ENSEMBLE & MODEL CLASS DEFINITIONS LOADED


In [4]:
# 4. LOADING DATA & MODELS
print("Loading Test Data...")
X_test = np.load(f"{FEATURE_DIR}/final_X_test.npy")
y_test = np.load(f"{FEATURE_DIR}/final_y_test.npy")

with open(f"{FEATURE_DIR}/dataset_metadata.pkl", "rb") as f:
    meta = pickle.load(f)
class_names = list(meta['class_to_idx'].keys())
NUM_CLASSES = len(class_names)

print(f"Test Data Shape: {X_test.shape}")
print(f"Classes: {class_names}")

print("\nLoading Pre-trained Models...")
# Helper to find the latest folder containing 'WeightStacker.pkl'
def find_latest_valid_folder(pattern, required_file):
    folders = glob.glob(os.path.join(BACKUP_DIR, pattern))
    folders.sort(key=os.path.getmtime, reverse=True)
    for folder in folders:
        if os.path.exists(os.path.join(folder, required_file)):
            return folder
    return None

# Locate the folder from Notebook 8 (Ensemble Run)
best_folder = find_latest_valid_folder("ensemble_run_*", "WeightStacker.pkl")

models = {}
if best_folder:
    print(f"   📂 Loading from: {best_folder}")

    # Load Stacker
    try:
        with open(f"{best_folder}/WeightStacker.pkl", "rb") as f:
            models['Stacking Ensemble'] = pickle.load(f)
        print("     ✅ Loaded WeightStacker.pkl")
    except Exception as e:
        print(f"     ❌ Failed to load WeightStacker: {e}")

    # Load Base Models (Check for both CamelCase and snake_case filenames)
    base_mappings = {
        "XGBoost": ["CustomXGB.pkl", "custom_xgb.pkl"],
        "LightGBM": ["CustomLGBM.pkl", "custom_lgbm.pkl"],
        "CatBoost": ["CustomCatB.pkl", "custom_catb.pkl"]
    }

    for model_name, filenames in base_mappings.items():
        loaded = False
        for fname in filenames:
            fpath = os.path.join(best_folder, fname)
            if os.path.exists(fpath):
                try:
                    with open(fpath, "rb") as f:
                        models[model_name] = pickle.load(f)
                    print(f"     ✅ Loaded {model_name} from {fname}")
                    loaded = True
                    break
                except Exception as e:
                    print(f"     ⚠️ Error loading {fname}: {e}")
        if not loaded:
            print(f"     ❌ Could not find file for {model_name}")
else:
    print("❌ Critical Error: No valid backup folder found! Make sure Notebook 8 ran successfully.")

Loading Test Data...
Test Data Shape: (548, 2048)
Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Loading Pre-trained Models...
   📂 Loading from: /content/drive/MyDrive/FabricDefect/backups/ensemble_run_20251205_014432
     ✅ Loaded WeightStacker.pkl
     ✅ Loaded XGBoost from CustomXGB.pkl
     ✅ Loaded LightGBM from CustomLGBM.pkl
     ✅ Loaded CatBoost from CustomCatB.pkl


In [5]:
# 5. EVALUATE ALL MODELS
print("\nRunning Evaluation on Test Set...")
results_store = {}
model_probs_list = []

# Check if base models are loaded
base_keys = ['XGBoost', 'LightGBM', 'CatBoost']
if all(k in models for k in base_keys):
    for name in base_keys:
        print(f"   Predicting with {name}...")
        model = models[name]
        # Get probabilities
        probs = model.predict_proba(X_test)
        model_probs_list.append(probs)

        # Get hard predictions
        pred = np.argmax(probs, axis=1)

        # Calculate Metrics
        metrics_dict = CustomEvaluator.classification_report(y_test, pred, class_names)
        cm = CustomEvaluator.confusion_matrix(y_test, pred, NUM_CLASSES)

        # Calculate One-vs-Rest ROC
        roc_data = {}
        for i in range(NUM_CLASSES):
             y_bin = (y_test == i).astype(int)
             y_score = probs[:, i]
             fpr, tpr, auc = CustomEvaluator.roc_curve_manual(y_bin, y_score)
             roc_data[class_names[i]] = {'fpr': fpr, 'tpr': tpr, 'auc': auc}

        results_store[name] = {'metrics': metrics_dict, 'cm': cm, 'roc': roc_data}

    # Evaluate Ensemble
    if 'Stacking Ensemble' in models:
        print(f"   Predicting with Stacking Ensemble...")
        try:
            # Ensemble takes list of prob arrays
            pred_ens = models['Stacking Ensemble'].predict(model_probs_list)
            probs_ens = models['Stacking Ensemble'].predict_proba(model_probs_list)

            metrics_dict = CustomEvaluator.classification_report(y_test, pred_ens, class_names)
            cm = CustomEvaluator.confusion_matrix(y_test, pred_ens, NUM_CLASSES)

            # ROC for Ensemble
            roc_data = {}
            for i in range(NUM_CLASSES):
                 y_bin = (y_test == i).astype(int)
                 y_score = probs_ens[:, i]
                 fpr, tpr, auc = CustomEvaluator.roc_curve_manual(y_bin, y_score)
                 roc_data[class_names[i]] = {'fpr': fpr, 'tpr': tpr, 'auc': auc}

            results_store['Stacking Ensemble'] = {'metrics': metrics_dict, 'cm': cm, 'roc': roc_data}
        except Exception as e:
            print(f"   ❌ Ensemble prediction error: {e}")
else:
    print("⚠️ Missing base models, cannot proceed with full evaluation.")


Running Evaluation on Test Set...
   Predicting with XGBoost...
Class                Precision  Recall     F1-Score   Support   
-----------------------------------------------------------------
0                    0.7143     0.5882     0.6452     17
1                    0.9847     0.9877     0.9862     326
2                    0.7308     0.7308     0.7308     52
3                    0.7083     0.7083     0.7083     24
4                    0.8049     0.8684     0.8354     38
5                    0.9556     0.9451     0.9503     91
-----------------------------------------------------------------
Macro Avg            0.8164     0.8048     0.8094     548
Weighted Avg         0.9228     0.9234     0.9228     548
   Predicting with LightGBM...
Class                Precision  Recall     F1-Score   Support   
-----------------------------------------------------------------
0                    0.6875     0.6471     0.6667     17
1                    0.9847     0.9847     0.9847     326
2 

/tmp/ipython-input-773195868.py:95: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc = np.trapz(tpr, fpr)


In [6]:
# 6. ADVANCED VISUALIZATION DASHBOARD (SVG)
print("Generating Advanced Dashboard...")

if results_store:
    # Find Best Model
    best_model_name = max(results_store, key=lambda x: results_store[x]['metrics']['accuracy'])
    print(f"🏆 Best Model: {best_model_name}")

    # Create Figure Layout
    fig = plt.figure(figsize=(20, 12), constrained_layout=True)
    gs = fig.add_gridspec(2, 3)

    # 1. Accuracy Comparison Bar Chart
    ax1 = fig.add_subplot(gs[0, 0])
    names = list(results_store.keys())
    accs = [results_store[n]['metrics']['accuracy'] for n in names]
    bars = ax1.bar(names, accs, color=['#3498db', '#e74c3c', '#2ecc71', '#9b59b6'])
    ax1.set_title("Model Accuracy Comparison", fontweight='bold')
    ax1.set_ylim(0, 1.1)
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height, f'{height:.2%}', ha='center', va='bottom')

    # 2. Confusion Matrix (Best Model)
    ax2 = fig.add_subplot(gs[0, 1])
    cm = results_store[best_model_name]['cm']
    im = ax2.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax2.set_title(f"Confusion Matrix: {best_model_name}", fontweight='bold')
    tick_marks = np.arange(len(class_names))
    ax2.set_xticks(tick_marks)
    ax2.set_yticks(tick_marks)
    ax2.set_xticklabels(class_names, rotation=45, ha="right")
    ax2.set_yticklabels(class_names)
    # Annotate
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax2.text(j, i, format(cm[i, j], 'd'), ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")

    # 3. ROC Curves (One-vs-Rest for Best Model)
    ax3 = fig.add_subplot(gs[0, 2])
    roc_data = results_store[best_model_name]['roc']
    for cls, data in roc_data.items():
        ax3.plot(data['fpr'], data['tpr'], lw=2, label=f'{cls} (AUC={data["auc"]:.2f})')
    ax3.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax3.set_xlim([0.0, 1.0])
    ax3.set_ylim([0.0, 1.05])
    ax3.set_xlabel('False Positive Rate')
    ax3.set_ylabel('True Positive Rate')
    ax3.set_title(f'ROC Curves: {best_model_name}', fontweight='bold')
    ax3.legend(loc="lower right", fontsize='small')

    # 4. F1-Score Per Class Comparison
    ax4 = fig.add_subplot(gs[1, :])
    x = np.arange(len(class_names))
    width = 0.2

    for i, (model_name, res) in enumerate(results_store.items()):
        f1s = [res['metrics']['per_class'][cls]['f1-score'] for cls in class_names]
        ax4.bar(x + i*width, f1s, width, label=model_name)

    ax4.set_xticks(x + width, class_names)
    ax4.set_title("F1-Score per Class by Model", fontweight='bold')
    ax4.legend()
    ax4.grid(axis='y', linestyle='--', alpha=0.7)

    # Save as SVG
    svg_path = os.path.join(METRICS_DIR, "advanced_evaluation_dashboard.svg")
    plt.savefig(svg_path, format='svg')
    print(f"✅ Dashboard saved to: {svg_path}")
    plt.show()
else:
    print("❌ No results to visualize.")

Generating Advanced Dashboard...
🏆 Best Model: Stacking Ensemble
✅ Dashboard saved to: /content/drive/MyDrive/FabricDefect/metrics/advanced_evaluation_dashboard.svg


In [7]:
# 7. EXPORT METRICS TO JSON (FIXED TYPE ERROR)
if results_store:
    print("Exporting metrics to JSON...")

    # Helper to handle numpy types
    def convert_to_native(obj):
        if isinstance(obj, (np.int64, np.int32)):
            return int(obj)
        elif isinstance(obj, (np.float64, np.float32)):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj

    export_data = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "winner": best_model_name,
        "models": {}
    }

    for name, data in results_store.items():
        # Calculate Macro AUC safely
        auc_scores = [v['auc'] for k,v in data['roc'].items()]
        macro_auc = np.mean(auc_scores) if auc_scores else 0.0

        export_data['models'][name] = {
            "overall_accuracy": float(data['metrics']['accuracy']),
            "macro_auc": float(macro_auc),
            "weighted_avg": {k: float(v) for k,v in data['metrics']['weighted_avg'].items()},
            "per_class_metrics": {}
        }

        # Per class details
        for cls_name in class_names:
            cls_key = str(cls_name) # Ensure string key
            stats = data['metrics']['per_class'][cls_name]
            roc = data['roc'][cls_name]

            export_data['models'][name]['per_class_metrics'][cls_key] = {
                "precision": float(stats['precision']),
                "recall": float(stats['recall']),
                "f1_score": float(stats['f1-score']),
                "auc": float(roc['auc'])
            }

    output_file = os.path.join(METRICS_DIR, "final_evaluation_report.json")
    with open(output_file, 'w') as f:
        json.dump(export_data, f, indent=4)

    print(f"✅ Detailed JSON exported to: {output_file}")

Exporting metrics to JSON...
✅ Detailed JSON exported to: /content/drive/MyDrive/FabricDefect/metrics/final_evaluation_report.json
